## DATA CLEANING AND INSPECTION
-loaded five raw datasets collected for the Travel Africa RAG Assistant 
and prepares them for the RAG pipeline.


| Dataset | Records | What It Contains |
|---------|---------|-----------------|
| Hotels | 95 | Hotels across Kenya, Tanzania, Uganda |
| Attractions | 25 | Parks, beaches, historical sites, natural wonders |
| Transport | 29 | Uber, Bolt, matatus, ferries, boda bodas per city |
| Scams | 20 | Common tourist scams and how to avoid them |
| Malls | 18 | Shopping centres with opening hours and highlights |

-Loaded all five CSV files and checked shapes and column names.Checked for missing values, none found across all datasets.Checked for duplicates, none found across all datasets.Standardised text, stripped whitespace from key columns. Created combined text, merged all relevant columns into one descriptive text field per record. This is what gets embedded into the vector database. The RAG system searches based on meaning not individual columns. By combining all relevant information into one rich text field we give the embedding model the full context it needs to find the right records for any user question.

In [1]:
import pandas as pd
import numpy as np

hotels = pd.read_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\raw\hotels.csv')
attractions = pd.read_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\raw\attractions.csv')
transport = pd.read_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\raw\transport.csv')
scams = pd.read_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\raw\scams.csv')
malls = pd.read_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\raw\malls.csv')

print("Hotels:", hotels.shape)
print("Attractions:", attractions.shape)
print("Transport:", transport.shape)
print("Scams:", scams.shape)
print("Malls:", malls.shape)

Hotels: (95, 12)
Attractions: (25, 10)
Transport: (29, 9)
Scams: (20, 6)
Malls: (18, 10)


In [2]:
for name, df in [('Hotels', hotels), ('Attractions', attractions), 
                  ('Transport', transport), ('Scams', scams), ('Malls', malls)]:
    print(f"\n{name} missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])



Hotels missing values:
Series([], dtype: int64)

Attractions missing values:
Series([], dtype: int64)

Transport missing values:
Series([], dtype: int64)

Scams missing values:
Series([], dtype: int64)

Malls missing values:
Series([], dtype: int64)


In [3]:
for name, df in [('Hotels', hotels), ('Attractions', attractions), 
                  ('Transport', transport), ('Scams', scams), ('Malls', malls)]:
    print(f"{name} duplicates: {df.duplicated().sum()}")

Hotels duplicates: 0
Attractions duplicates: 0
Transport duplicates: 0
Scams duplicates: 0
Malls duplicates: 0


In [4]:
hotels = hotels.drop_duplicates()
hotels['hotel_name'] = hotels['hotel_name'].str.strip()
hotels['location'] = hotels['location'].str.strip()
hotels['country'] = hotels['country'].str.strip()
hotels['rating'] = pd.to_numeric(hotels['rating'], errors='coerce')
hotels['rating'] = hotels['rating'].fillna(hotels['rating'].median())
hotels['price_range'] = hotels['price_range'].fillna('Price on request')
hotels['description'] = hotels['description'].fillna('No description available')
print("Hotels cleaned:", hotels.shape)

Hotels cleaned: (95, 12)


In [5]:
attractions = attractions.drop_duplicates()
attractions['name'] = attractions['name'].str.strip()
attractions['location'] = attractions['location'].str.strip()

transport = transport.drop_duplicates()
transport['name'] = transport['name'].str.strip()

scams = scams.drop_duplicates()
scams['scam_name'] = scams['scam_name'].str.strip()

malls = malls.drop_duplicates()
malls['name'] = malls['name'].str.strip()

print("All datasets cleaned")

All datasets cleaned


In [ ]:
#  combined text column for each dataset, for embedding into the RAG system
hotels['combined_text'] = (
    "Hotel: " + hotels['hotel_name'] + ". " +
    "Location: " + hotels['location'] + ", " + hotels['country'] + ". " +
    "Category: " + hotels['category'] + ". " +
    "Description: " + hotels['description'] + ". " +
    "Price range: " + hotels['price_range'] + ". " +
    "Amenities: " + hotels['amenities'].str.replace('|', ', ') + ". " +
    "Room types: " + hotels['room_types'].str.replace('|', ', ') + ". " +
    "Rating: " + hotels['rating'].astype(str) + " out of 5. " +
    "Nearby attractions: " + hotels['nearby_attractions'].str.replace('|', ', ') + "."
)

attractions['combined_text'] = (
    "Attraction: " + attractions['name'] + ". " +
    "Location: " + attractions['location'] + ", " + attractions['country'] + ". " +
    "Type: " + attractions['type'] + ". " +
    "Description: " + attractions['description'] + ". " +
    "Price: " + attractions['price_range'] + ". " +
    "Opening hours: " + attractions['opening_hours'] + ". " +
    "Tips: " + attractions['tips'] + "."
)

transport['combined_text'] = (
    "Transport in " + transport['city'] + ", " + transport['country'] + ": " +
    transport['name'] + ". " +
    "Type: " + transport['transport_type'] + ". " +
    "Description: " + transport['description'] + ". " +
    "Cost: " + transport['approximate_cost'] + ". " +
    "How to use: " + transport['how_to_use'] + ". " +
    "Tips: " + transport['tips'] + "."
)

scams['combined_text'] = (
    "Scam warning in " + scams['location'] + ", " + scams['country'] + ": " +
    scams['scam_name'] + ". " +
    "Description: " + scams['description'] + ". " +
    "How to avoid: " + scams['how_to_avoid'] + ". " +
    "What to do: " + scams['what_to_do_if_it_happens'] + "."
)

malls['combined_text'] = (
    "Mall: " + malls['name'] + ". " +
    "Location: " + malls['location'] + ", " + malls['country'] + ". " +
    "Description: " + malls['description'] + ". " +
    "Opening hours: " + malls['opening_hours'] + ". " +
    "Highlights: " + malls['highlights'].str.replace('|', ', ') + ". " +
    "Tips: " + malls['tips'] + "."
)

print("Combined text created for all datasets")
print("\nSample hotel text:")
print(hotels['combined_text'].iloc[0])

Combined text created for all datasets

Sample hotel text:
Hotel: Sarova Stanley. Location: Nairobi, Kenya. Category: Luxury. Description: Historic luxury hotel in the heart of Nairobi CBD established in 1902. Price range: 150-400 USD. Amenities: WiFi, Pool, Restaurant, Gym, Spa, Bar, Parking. Room types: Standard, Deluxe, Suite. Rating: 4.5 out of 5. Nearby attractions: Nairobi National Park, CBD Shopping.


In [7]:
hotels.to_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\cleaned\hotels_cleaned.csv', index=False)
attractions.to_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\cleaned\attractions_cleaned.csv', index=False)
transport.to_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\cleaned\transport_cleaned.csv', index=False)
scams.to_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\cleaned\scams_cleaned.csv', index=False)
malls.to_csv(r'C:\Users\User\Desktop\travel-africa-rag\data\cleaned\malls_cleaned.csv', index=False)
print("All cleaned files saved")

All cleaned files saved
